# DermaScope — Clasificador con y sin CBAM

Entrena las dos corridas que sostienen §4.1 (clasificación) y la ablación de §4.3
(CBAM), cuantiza el mejor modelo a INT8 (§4.4) y mide los tiempos de GPU de §4.5.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.
Si no seleccionas GPU, la celda de verificación se detiene y te avisa.

El dataset se descarga directo de Harvard Dataverse (~2,6 GB, unos 2 minutos con la
conexión de Colab). No hace falta subir nada a Drive.

Duración aproximada de la corrida completa en T4: **2 a 2,5 horas** (unos 40 min por
entrenamiento, más la evaluación ONNX de la cuantización).

**Importante sobre las sesiones de Colab.** La máquina se borra al desconectarse. La
última celda copia los checkpoints y las métricas a tu Drive; no cierres la pestaña
sin ejecutarla o perderás el entrenamiento.


## 1. Verificar la GPU

In [ ]:
import torch, sys

if not torch.cuda.is_available():
    raise SystemExit(
        'No hay GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU T4, '
        'y vuelve a ejecutar esta celda.'
    )

print('GPU        :', torch.cuda.get_device_name(0))
print('VRAM       :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('torch      :', torch.__version__)
print('CUDA       :', torch.version.cuda)
print('Python     :', sys.version.split()[0])
print()
print('Anota el nombre de la GPU: va en la tabla de §4.5 del informe.')

## 2. Traer el código

Pon la URL de tu repositorio en `REPO_URL`. Si el repo es privado, Colab pedirá
credenciales; la alternativa es hacerlo público, que para un trabajo académico suele
estar bien.

In [ ]:
REPO_URL = 'https://github.com/USUARIO/dermascope.git'  # <-- reemplazar

import os, subprocess
from pathlib import Path

if 'USUARIO' in REPO_URL:
    raise SystemExit(
        'Falta reemplazar REPO_URL por la URL real del repositorio. '
        'Sin eso el clone falla a mitad de la celda y el error no dice por que.'
    )

REPO_DIR = Path('/content/dermascope')
if REPO_DIR.exists():
    # Re-ejecutable: si ya se clono, solo actualiza.
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('
directorio:', Path.cwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


## 3. Dependencias

Colab ya trae torch, torchvision, pandas, sklearn y matplotlib. Solo faltan estas, y a
propósito **no** se instala desde `requirements.txt`: reinstalar torch sobre el de Colab
rompe la compatibilidad con CUDA de la imagen.

In [ ]:
!pip install --quiet transformers segmentation-models-pytorch albumentations onnx onnxruntime

import albumentations, transformers
print('albumentations', albumentations.__version__)
print('transformers  ', transformers.__version__)

In [ ]:
# Verificacion del entorno antes de gastar tiempo de GPU: 9 comprobaciones que no
# necesitan el dataset. Si alguna falla, el problema es de versiones, no de datos.
!python scripts/smoke_test.py

## 4. Descargar el dataset

Fuente: Harvard Dataverse, DOI `10.7910/DVN/DBW86T`. Descarga anónima, sin cuenta.
Se usa `curl` y no `requests` porque escribe directo a disco con memoria constante.

In [ ]:
%%bash
set -e
RAW=/content/data/raw
mkdir -p $RAW

dl() {  # dl <id> <destino>
  if [ -s "$2" ]; then echo "ya existe: $(basename $2)"; return; fi
  curl -sS -L --retry 5 --retry-delay 3 -C - \
    -o "$2" "https://dataverse.harvard.edu/api/access/datafile/$1"
  echo "$(basename $2): $(du -h $2 | cut -f1)"
}

dl 4338392 $RAW/HAM10000_metadata.tab
dl 3838943 $RAW/segmentations.zip
dl 3172585 $RAW/HAM10000_images_part_1.zip
dl 3172584 $RAW/HAM10000_images_part_2.zip

ls -lh $RAW

In [ ]:
# Extraccion reanudable, salta lo ya extraido comparando tamano.
!python scripts/extract_data.py /content/data/raw/HAM10000_images \
    /content/data/raw/HAM10000_images_part_1.zip \
    /content/data/raw/HAM10000_images_part_2.zip

!python scripts/extract_data.py /content/data/raw/HAM10000_segmentations \
    /content/data/raw/segmentations.zip

## 5. Splits

Se escribe un `paths.local.yaml` apuntando a `/content/data`. Ese archivo está en
`.gitignore`, así que no interfiere con la configuración de nadie más, y es el mismo
mecanismo que se usa en local para sacar los datos de OneDrive.

In [ ]:
from pathlib import Path

Path('configs/paths.local.yaml').write_text(
    'paths:\n'
    '  ham_metadata: /content/data/raw/HAM10000_metadata.tab\n'
    '  ham_images: /content/data/raw/HAM10000_images\n'
    '  seg_masks: /content/data/raw/HAM10000_segmentations\n'
    '  processed: /content/data/processed\n'
    '  models: /content/models\n',
    encoding='utf-8',
)
print(Path('configs/paths.local.yaml').read_text())

In [ ]:
!python -m src.data.build_splits --config configs/paths.yaml

## 6. Entrenar — con CBAM

ResNet-34 preentrenada en ImageNet, con CBAM inyectado tras `layer3` y `layer4`.
El checkpoint se elige por `val_macro_f1`, no por accuracy: con `nv` al 67% del dataset,
la accuracy sube sola sin que el modelo aprenda las clases raras.

In [ ]:
!python -m src.train.train_classifier --config configs/classification.yaml

## 7. Entrenar — sin CBAM (ablación)

Misma semilla, mismo split, mismos hiperparámetros. El único factor que cambia es el
bloque de atención, y eso es lo que hace que la diferencia en macro-F1 sea atribuible a
CBAM y no al ruido de entrenamiento. Sin esta corrida no se puede responder lo que
pide §4.3.

In [ ]:
!python -m src.train.train_classifier --config configs/classification.yaml --no-cbam

## 8. Comparar la ablación

In [ ]:
!python -m src.eval.compare_ablation --cls-config configs/classification.yaml

## 9. Cuantización INT8 (§4.4)

Se exporta el clasificador con CBAM a ONNX y se cuantiza a INT8 con calibración
estática sobre 200 imágenes reales de train. La calibración necesita imágenes reales
y no ruido: los rangos de activación que se estiman son los que decide el dataset.

El script evalúa FP32 e INT8 sobre el mismo test y escribe `optimization.json` con
tamaño en disco, accuracy, macro-F1 y la caída de F1 frente a la tolerancia de
`max_f1_drop` del YAML. Si la caída se pasa, la ruta declarada es la poda
(`src/optimize/prune.py`).

La evaluación corre en CPU con ONNX Runtime, así que esta celda tarda unos minutos
aunque haya GPU: es inferencia sobre las 1.502 imágenes de test, dos veces.


In [ ]:
!python -m src.optimize.quantize --config configs/classification.yaml


## 10. Tiempos de GPU (§4.5)

La medición de CPU se hace en la máquina local, con su propio hardware anotado. Aquí
solo la GPU. El script registra automáticamente el nombre de la GPU y las versiones.

In [ ]:
!python -m src.eval.benchmark --device cuda --batch-sizes 1 8

## 11. Guardar todo en Drive

**No cierres la sesión sin ejecutar esto.** Colab borra la máquina al desconectarse.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

destino = Path('/content/drive/MyDrive/dermascope')
(destino / 'models').mkdir(parents=True, exist_ok=True)
(destino / 'results').mkdir(parents=True, exist_ok=True)

copiados = []
# Los .onnx entran junto a los .pt: sin ellos no se puede rehacer la tabla de §4.4
# ni medir el INT8 en la CPU local, que es donde esa medicion tiene sentido.
for patron in ('*.pt', '*.onnx'):
    for ckpt in Path('/content/models').glob(patron):
        shutil.copy2(ckpt, destino / 'models' / ckpt.name)
        copiados.append(f'{ckpt.name}  ({ckpt.stat().st_size / 1e6:.1f} MB)')

for res in Path('reports/results').glob('*'):
    if res.is_file():
        shutil.copy2(res, destino / 'results' / res.name)
        copiados.append(res.name)

print(f'Copiado a {destino}:')
for c in copiados:
    print('  ', c)
print('
Descarga la carpeta results/ y pasasela a Claude Code para que actualice')
print('la model card y el informe con las metricas reales.')
